# Person Presence Detection

Purpose: compare empty-room and person-present CSI traces with deterministic synthetic data so the visual lab is usable before hardware replay helpers land.

Run path: install research extras with `uv sync --extra research`, open this notebook, and choose Run All. The notebook uses optional simulator fixtures when present and falls back to inline NumPy fixtures otherwise.

Fixture / simulated source: `load_capture_pair` first tries guarded simulator fixture imports from `ruview.hardware.simulator`. If those imports or calls fail, `inline_room_capture` generates a 3-stream, 56-subcarrier capture pair with shared static multipath and a person-present reflection envelope.


In [ ]:
# Data source option: synthetic fixture or local ESP32 CSI recording.
USE_RECORDING = False
RECORDING_CSV = None  # Set to a specific *_csi.csv path, or leave None to auto-pick from data/recordings.
_RECORDING_MAX_FRAMES = None

from pathlib import Path
import numpy as np

try:
    from ruview.hardware.esp32_capture_analysis import load_esp32_capture
except ImportError:  # Allows the notebook to render in environments without the package installed yet.
    load_esp32_capture = None


def _find_repo_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'ruview').exists():
            return candidate
    return current


def _pick_recording_csv(repo_root, override=None):
    if override:
        return Path(override).expanduser().resolve()
    candidates = sorted((repo_root / 'data' / 'recordings').glob('**/*_csi.csv'))
    return candidates[0] if candidates else None


def _elapsed_seconds_for_capture(capture):
    real_ts = np.asarray(capture.timestamps, dtype=np.float64)
    if np.isfinite(real_ts).sum() >= 2 and float(np.nanmax(real_ts) - np.nanmin(real_ts)) > 0:
        return real_ts - float(np.nanmin(real_ts))
    mono = np.asarray(capture.host_monotonic_ns, dtype=np.float64)
    if mono.size == 0:
        return np.array([], dtype=np.float64)
    return (mono - float(mono[0])) / 1_000_000_000.0


def _fill_invalid(values, mask):
    filled = np.asarray(values, dtype=np.float64).copy()
    filled[~mask] = np.nan
    valid = np.isfinite(filled)
    counts = valid.sum(axis=0)
    sums = np.where(valid, filled, 0.0).sum(axis=0)
    col_means = np.divide(sums, counts, out=np.zeros_like(sums), where=counts > 0)
    rows, cols = np.where(~valid)
    filled[rows, cols] = col_means[cols]
    return filled


def _capture_to_csi_tensor(capture, max_frames=None):
    take = slice(None) if max_frames is None else slice(0, max_frames)
    amp = _fill_invalid(capture.amplitude[take], capture.valid_mask[take])
    phase = _fill_invalid(capture.phase[take], capture.valid_mask[take])
    time_s = _elapsed_seconds_for_capture(capture)[take]
    csi = (amp * np.exp(1j * phase))[:, None, :]
    return csi, time_s


def _effective_sample_rate(time_s):
    time_s = np.asarray(time_s, dtype=np.float64)
    duration = float(time_s[-1] - time_s[0]) if time_s.size >= 2 else 0.0
    return float((time_s.size - 1) / duration) if duration > 0 else 1.0


_recording_repo_root = _find_repo_root()
_recording_path = _pick_recording_csv(_recording_repo_root, RECORDING_CSV)
_recording_capture = None
_recording_csi = None
_recording_time_s = None

if USE_RECORDING:
    if load_esp32_capture is None:
        raise ImportError('ruview.hardware.esp32_capture_analysis.load_esp32_capture is required for recordings')
    if _recording_path is None:
        raise FileNotFoundError('No *_csi.csv recording found under data/recordings; set RECORDING_CSV explicitly.')
    _recording_capture = load_esp32_capture(_recording_path)
    _recording_csi, _recording_time_s = _capture_to_csi_tensor(_recording_capture, _RECORDING_MAX_FRAMES)
    print(f'Using recording: {_recording_path} ({_recording_csi.shape[0]} frames, {_recording_csi.shape[2]} subcarriers)')
else:
    print('Using synthetic fixture. Set USE_RECORDING=True to plot from a local *_csi.csv recording.')


import numpy as np

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    raise RuntimeError('Install the research extra with: uv sync --extra research') from exc

try:
    from ruview.hardware.simulator import empty_room_fixture, person_present_fixture
except Exception as exc:
    empty_room_fixture = person_present_fixture = None
    SIMULATOR_IMPORT_ERROR = exc
else:
    SIMULATOR_IMPORT_ERROR = None


def inline_room_capture(condition, frames=90, streams=3, subcarriers=56, duration_s=12.0, seed=101):
    if condition not in {'empty', 'present'}:
        raise ValueError('condition must be empty or present')

    rng = np.random.default_rng(seed + (37 if condition == 'present' else 0))
    time_s = np.linspace(0.0, duration_s, frames)
    subcarrier_axis = np.linspace(-1.0, 1.0, subcarriers)
    capture = np.empty((frames, streams, subcarriers), dtype=np.complex128)

    for frame_index, time_value in enumerate(time_s):
        slow_drift = 0.01 * np.sin(2.0 * np.pi * 0.04 * time_value)
        respiration = 0.035 * np.sin(2.0 * np.pi * 0.25 * time_value)
        motion_phase = 0.08 * np.sin(2.0 * np.pi * 0.11 * time_value + 2.0 * np.pi * subcarrier_axis)
        reflection = 0.18 * np.exp(-((subcarrier_axis - 0.12) / 0.24) ** 2)

        for stream in range(streams):
            static_amp = 1.0 + 0.04 * np.sin(2.0 * np.pi * (stream + 1) * (subcarrier_axis + 1.2))
            amplitude = static_amp + slow_drift + rng.normal(0.0, 0.006, subcarriers)
            phase = 0.04 * stream + 0.02 * np.sin(2.0 * np.pi * 0.03 * time_value + subcarrier_axis)

            if condition == 'present':
                amplitude = amplitude + reflection + respiration
                phase = phase + motion_phase

            phase = phase + rng.normal(0.0, 0.004, subcarriers)
            capture[frame_index, stream] = amplitude * np.exp(1j * phase)

    return capture


def load_capture_pair():
    if empty_room_fixture is not None and person_present_fixture is not None:
        try:
            empty = np.asarray(empty_room_fixture(), dtype=np.complex128)
            present = np.asarray(person_present_fixture(), dtype=np.complex128)
            if empty.ndim == 3 and present.shape == empty.shape:
                return empty, present, 'ruview.hardware.simulator fixtures'
        except Exception:
            pass

    return inline_room_capture('empty'), inline_room_capture('present'), 'inline synthetic fallback'


def recording_presence_pair(csi, path_label):
    segment = max(8, min(csi.shape[0] // 3, 120))
    if csi.shape[0] < segment * 2:
        midpoint = csi.shape[0] // 2
        return csi[:midpoint], csi[midpoint:], f'recording split: {path_label}'
    amp = np.abs(csi[:, 0, :])
    frame_motion = np.r_[0.0, np.nanmean(np.square(np.diff(amp, axis=0)), axis=1)]
    rolling_motion = np.convolve(frame_motion, np.ones(segment, dtype=float), mode='same')
    active_center = int(np.nanargmax(rolling_motion))
    active_start = max(0, min(csi.shape[0] - segment, active_center - segment // 2))
    empty = csi[:segment]
    present = csi[active_start:active_start + segment]
    return empty, present, f'recording baseline vs highest-motion segment: {path_label}'


if USE_RECORDING and _recording_csi is not None:
    empty, present, source = recording_presence_pair(_recording_csi, str(_recording_path))
else:
    empty, present, source = load_capture_pair()
time_s = np.linspace(0.0, 12.0, min(empty.shape[0], present.shape[0]))
empty = empty[:time_s.size]
present = present[:time_s.size]
empty_amp = np.abs(empty)
present_amp = np.abs(present)
empty_phase = np.unwrap(np.angle(empty), axis=2)
present_phase = np.unwrap(np.angle(present), axis=2)

metrics = {
    'source': source,
    'shape': empty.shape,
    'empty_mean_amplitude': round(float(empty_amp.mean()), 4),
    'present_mean_amplitude': round(float(present_amp.mean()), 4),
    'delta_mean_amplitude': round(float(present_amp.mean() - empty_amp.mean()), 4),
    'empty_amplitude_variance': round(float(empty_amp.var()), 6),
    'present_amplitude_variance': round(float(present_amp.var()), 6),
}

metrics


In [ ]:
empty_mean_by_time = empty_amp.mean(axis=(1, 2))
present_mean_by_time = present_amp.mean(axis=(1, 2))
empty_var_by_time = empty_amp.var(axis=(1, 2))
present_var_by_time = present_amp.var(axis=(1, 2))
delta_by_time_subcarrier = present_amp.mean(axis=1) - empty_amp.mean(axis=1)
empty_phase_std = empty_phase.std(axis=(1, 2))
present_phase_std = present_phase.std(axis=(1, 2))

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

axes[0, 0].plot(time_s, empty_mean_by_time, label='empty room')
axes[0, 0].plot(time_s, present_mean_by_time, label='person present')
axes[0, 0].set_title('Mean amplitude over time')
axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('Mean amplitude')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend(loc='best')

axes[0, 1].plot(time_s, empty_var_by_time, label='empty room')
axes[0, 1].plot(time_s, present_var_by_time, label='person present')
axes[0, 1].set_title('Amplitude variance over time')
axes[0, 1].set_xlabel('Time (s)')
axes[0, 1].set_ylabel('Variance')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend(loc='best')

image = axes[1, 0].imshow(
    delta_by_time_subcarrier.T,
    aspect='auto',
    origin='lower',
    extent=[time_s[0], time_s[-1], 0, empty.shape[2] - 1],
    cmap='coolwarm',
)
axes[1, 0].set_title('Person minus empty amplitude')
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('Subcarrier index')
fig.colorbar(image, ax=axes[1, 0], label='Delta amplitude')

axes[1, 1].plot(time_s, empty_phase_std, label='empty room')
axes[1, 1].plot(time_s, present_phase_std, label='person present')
axes[1, 1].set_title('Unwrapped phase spread')
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel('Standard deviation (radians)')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend(loc='best')

plt.show()


Expected interpretation: the person-present trace should show a higher mean amplitude near the synthetic reflection band, modest time-varying amplitude from respiration, and larger phase spread than the empty-room baseline.

Limitations: the fallback fixture is a small visual sanity check, not a validated room model. It does not include real ESP32 packet timing, antenna geometry, calibration drift, body pose, occlusion, or environmental motion.
